In [2]:
# -*- coding: utf-8 -*-
"""CALCULODEVOLATILIDADE-HARRV.ipynb"""

import pandas as pd, numpy as np, os, math
from openpyxl.utils import get_column_letter
from openpyxl.styles import Font, Alignment, PatternFill
from openpyxl import load_workbook

path="/content/PREÇO MED DIARIO PLD VERTICAL PERIODO 17-04-18 A 03-06-25.xlsx"
df=pd.read_excel(path)

# try to detect date and price columns
cols=list(df.columns)
date_col=None
price_col=None
for c in cols:
    cl=str(c).strip().lower()
    if "data" in cl:
        date_col=c
    if "pre" in cl and "m" in cl:
        price_col=c

if date_col is None:
    for c in cols:
        if str(c).strip().lower() in ["data","date"]:
            date_col=c; break

if price_col is None:
    for c in cols:
        if str(c).strip().lower() in ["preço","preco","pld","price","preço medio diário","preco medio diario"]:
            price_col=c; break

if date_col is None or price_col is None:
    raise ValueError(f"Não consegui identificar colunas de data/preço. Colunas: {cols}")

df=df.rename(columns={date_col:"data", price_col:"preço_PLD"})
df["data"]=pd.to_datetime(df["data"])
df=df.sort_values("data").reset_index(drop=True)

# log returns
df["retorno_log"]=np.log(df["preço_PLD"]/df["preço_PLD"].shift(1))

# realized volatility
df["volatilidade_realizada_diária"]=df["retorno_log"].abs()

# realized variance
df["RV"]=df["retorno_log"]**2

# HAR predictors
df["RV_d"]=df["RV"]
df["RV_w"]=df["RV"].rolling(window=5).mean()
df["RV_m"]=df["RV"].rolling(window=21).mean()

# regressão HAR-RV
reg=df[["RV_d","RV_w","RV_m","RV"]].copy()
reg["y"]=df["RV"].shift(-1)

X=reg[["RV_d","RV_w","RV_m"]]
y=reg["y"]

mask=~(X.isna().any(axis=1) | y.isna())
Xn=X[mask].values
yn=y[mask].values

Xmat=np.column_stack([np.ones(len(Xn)), Xn])

beta=np.linalg.lstsq(Xmat, yn, rcond=None)[0]

c, b_d, b_w, b_m = beta

yhat=Xmat@beta
resid=yn-yhat

ssr=np.sum(resid**2)
sst=np.sum((yn-np.mean(yn))**2)

r2=1-ssr/sst if sst>0 else np.nan
nobs=len(yn)

# forecast 1 dia
df["RV_HAR_forecast_1d"]=np.nan

for i in range(len(df)-1):

    rv_d=df.loc[i,"RV_d"]
    rv_w=df.loc[i,"RV_w"]
    rv_m=df.loc[i,"RV_m"]

    if np.isnan(rv_d) or np.isnan(rv_w) or np.isnan(rv_m):
        continue

    f=c + b_d*rv_d + b_w*rv_w + b_m*rv_m

    df.loc[i,"RV_HAR_forecast_1d"]=max(f, 1e-18)

# volatilidade diária
df["volatilidade_HAR-RV_diária"]=np.sqrt(df["RV_HAR_forecast_1d"])


# =============================
# NOVO CÁLCULO DOS HORIZONTES
# =============================

for n, lab in [(5,"1W"),(21,"1M"),(252,"1Y")]:

    df[f"volatilidade_HAR-RV_{lab}"] = (
        df["volatilidade_HAR-RV_diária"] * math.sqrt(n)
    )

    df[f"volatilidade_HAR-RV_{lab}_anualizada"] = (
        df["volatilidade_HAR-RV_diária"] * math.sqrt(252)
    )


# colunas finais
out=df[[
"data",
"preço_PLD",
"retorno_log",
"volatilidade_realizada_diária",
"volatilidade_HAR-RV_diária",
"volatilidade_HAR-RV_1W",
"volatilidade_HAR-RV_1M",
"volatilidade_HAR-RV_1Y",
"volatilidade_HAR-RV_1W_anualizada",
"volatilidade_HAR-RV_1M_anualizada",
"volatilidade_HAR-RV_1Y_anualizada"
]].copy()


# Excel output
out_path="/content/PLD_volatilidades_HAR-RV.xlsx"

with pd.ExcelWriter(out_path, engine="openpyxl") as writer:

    out.to_excel(writer, index=False, sheet_name="HAR-RV")

    params=pd.DataFrame({
        "Item":[
        "Modelo (HAR-RV)",
        "Equação",
        "Amostra (nobs)",
        "R² (OLS)",
        "c",
        "β_d",
        "β_w",
        "β_m"],
        "Valor":[
            "RV_{t+1} = c + β_d RV_t + β_w RV̄_{t-4:t} + β_m RV̄_{t-20:t} + ε_{t+1}",
            "RV_t = (retorno_log_t)^2",
            int(nobs),
            r2,
            c,b_d,b_w,b_m
        ]
    })

    params.to_excel(writer, index=False, sheet_name="Parâmetros")


# formatação
wb=load_workbook(out_path)

ws=wb["HAR-RV"]

ws.freeze_panes="A2"

header_fill=PatternFill("solid", fgColor="1F4E79")
header_font=Font(color="FFFFFF", bold=True)

for j,cell in enumerate(ws[1], start=1):

    cell.fill=header_fill
    cell.font=header_font
    cell.alignment=Alignment(horizontal="center", vertical="center", wrap_text=True)

    ws.column_dimensions[get_column_letter(j)].width=22 if j>1 else 14


for cell in ws["A"][1:]:
    cell.number_format="dd/mm/yyyy"


ws2=wb["Parâmetros"]

ws2.freeze_panes="A2"

for cell in ws2[1]:

    cell.fill=header_fill
    cell.font=header_font
    cell.alignment=Alignment(horizontal="center")

ws2.column_dimensions["A"].width=26
ws2.column_dimensions["B"].width=90

wb.save(out_path)

out_path

'/content/PLD_volatilidades_HAR-RV.xlsx'